# Baseline SmolVLA Evaluation (No Fine-Tuning)

Evaluate the **pretrained SmolVLA base model** (`lerobot/smolvla_base`) on the same
6 robosuite scenarios used for the fine-tuned model evaluation.

This provides a **baseline comparison** — the pretrained model has never seen our
Lift training data, so any difference in performance shows the effect of fine-tuning.

## Scenarios
| Scenario | Task | Max Steps |
|----------|------|-----------|
| Lift | Pick up the cube and lift it off the table | 400 |
| Stack | Pick up the red cube and stack it on top of the green cube | 500 |
| PickPlaceSingle | Pick up the object and place it in the bin | 500 |
| NutAssemblySquare | Pick up the square nut and place it on the square peg | 600 |
| NutAssemblyRound | Pick up the round nut and place it on the round peg | 600 |
| NutAssembly | Assemble both nuts onto their respective pegs | 800 |

**Runtime:** GPU (T4 sufficient for inference)

---
## 1. System Setup & Dependencies

In [ ]:
%%bash
# Install system dependencies for headless MuJoCo rendering
apt-get update -qq
apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev libglfw3 ffmpeg patchelf > /dev/null 2>&1

# Create NVIDIA EGL ICD config
mkdir -p /usr/share/glvnd/egl_vendor.d
cat > /usr/share/glvnd/egl_vendor.d/10_nvidia.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF

echo "System dependencies installed."

In [ ]:
# Install Python packages
# 1) Simulation + data
!pip install -q robosuite imageio[ffmpeg] matplotlib h5py Pillow pandas

# 2) Install LeRobot WITH [smolvla] extra from source
!pip install -q "lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git"

# 3) Pin numpy
!pip install -q numpy==2.0.2

print("\nPackages installed. Restarting runtime to fix numpy C bindings...")
print("After restart, SKIP this cell and continue from the next section.")

import os
os.kill(os.getpid(), 9)

### After runtime restart — continue from here

In [ ]:
import os

# MUST be set BEFORE importing mujoco or robosuite
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"

import robosuite
macros_private = os.path.join(os.path.dirname(robosuite.__file__), "macros_private.py")
if not os.path.exists(macros_private):
    with open(macros_private, "w") as f:
        f.write("# Auto-generated private macros\n")

import numpy as np
import robosuite as suite
import imageio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Video, display, HTML
import base64
import time
import json
import torch
from PIL import Image as PILImage

print(f"robosuite {robosuite.__version__}")
print(f"numpy {np.__version__}")
print(f"torch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Setup complete.")

---
## 2. Scenario Configuration

Same 6 scenarios as the fine-tuned evaluation.

In [ ]:
# ============================================================
# Task descriptions for each scenario (used by SmolVLA)
# ============================================================
TASK_DESCRIPTIONS = {
    "Lift": "Pick up the cube and lift it off the table",
    "Stack": "Pick up the red cube and stack it on top of the green cube",
    "PickPlaceSingle": "Pick up the object and place it in the bin",
    "NutAssemblySquare": "Pick up the square nut and place it on the square peg",
    "NutAssemblyRound": "Pick up the round nut and place it on the round peg",
    "NutAssembly": "Assemble both the square nut and round nut onto their respective pegs",
}

SCENARIOS = {
    "Lift": {
        "env_kwargs": dict(
            env_name="Lift", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=400,
        ),
        "max_steps": 400,
    },
    "Stack": {
        "env_kwargs": dict(
            env_name="Stack", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=500,
        ),
        "max_steps": 500,
    },
    "PickPlaceSingle": {
        "env_kwargs": dict(
            env_name="PickPlaceSingle", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=500,
        ),
        "max_steps": 500,
    },
    "NutAssemblySquare": {
        "env_kwargs": dict(
            env_name="NutAssemblySquare", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=600,
        ),
        "max_steps": 600,
    },
    "NutAssemblyRound": {
        "env_kwargs": dict(
            env_name="NutAssemblyRound", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=600,
        ),
        "max_steps": 600,
    },
    "NutAssembly": {
        "env_kwargs": dict(
            env_name="NutAssembly", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, single_object_mode=0, horizon=800,
        ),
        "max_steps": 800,
    },
}

print(f"Configured {len(SCENARIOS)} scenarios:")
for name, cfg in SCENARIOS.items():
    print(f"  {name}: max_steps={cfg['max_steps']}")

---
## 3. Load Pretrained SmolVLA Base Model (No Fine-Tuning)

Load `lerobot/smolvla_base` — the pretrained SmolVLA model trained on 10M frames
from 487 community robotics datasets. This model has **NOT** been fine-tuned on our
Lift training data.

In [ ]:
# ============================================================
# LOAD PRETRAINED SmolVLA BASE MODEL (NO FINE-TUNING)
# ============================================================
BASELINE_CHECKPOINT = "lerobot/smolvla_base"  # HuggingFace pretrained model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def load_vla_policy(checkpoint_path, device="cuda"):
    """Load SmolVLA policy with preprocessor/postprocessor pipelines.

    Returns (policy, preprocess, postprocess) tuple.
    The preprocessor handles language tokenization and state normalization.
    The postprocessor handles action unnormalization.
    """
    try:
        from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
        from lerobot.policies.factory import make_pre_post_processors

        policy = SmolVLAPolicy.from_pretrained(checkpoint_path)
        policy.to(device)
        policy.eval()

        preprocess, postprocess = make_pre_post_processors(
            policy.config,
            checkpoint_path,
            preprocessor_overrides={"device_processor": {"device": str(device)}},
        )

        print(f"Loaded VLA from {checkpoint_path}")
        print(f"  Preprocessor steps: {[type(s).__name__ for s in preprocess.steps]}")
        print(f"  Postprocessor steps: {[type(s).__name__ for s in postprocess.steps]}")
        return policy, preprocess, postprocess
    except Exception as e:
        import traceback
        print(f"Failed to load VLA: {e}")
        traceback.print_exc()
        print("Using random policy for testing.")
        return None, None, None


vla_policy, vla_preprocess, vla_postprocess = load_vla_policy(
    BASELINE_CHECKPOINT, device=DEVICE
)

---
## 4. Evaluation Helpers

In [ ]:
# ============================================================
# VLA EVALUATION HELPERS
# ============================================================


def get_vla_action(policy, preprocess, postprocess, obs, task_language, device="cuda"):
    """Get action from SmolVLA given robosuite observation.

    Pipeline:
      1. Build raw observation dict with LeRobot-compatible keys
      2. Convert to tensors, normalize images to [0,1], add batch dim
      3. Preprocess: tokenize language, normalize state (MEAN_STD)
      4. Model inference via select_action
      5. Postprocess: unnormalize action (MEAN_STD)
    """
    if policy is None:
        return np.random.uniform(-0.3, 0.3, size=7)

    from lerobot.policies.utils import prepare_observation_for_inference

    # Camera 1: agentview (flip vertical for MuJoCo -> standard orientation)
    agentview = np.flip(
        obs.get("agentview_image", np.zeros((256, 256, 3), dtype=np.uint8)), axis=0
    ).copy()
    if agentview.shape[0] != 256:
        agentview = np.array(PILImage.fromarray(agentview).resize((256, 256)))

    # Camera 2: wrist camera
    wrist = obs.get("robot0_eye_in_hand_image", None)
    if wrist is not None:
        wrist = np.flip(wrist, axis=0).copy()
        if wrist.shape[0] != 256:
            wrist = np.array(PILImage.fromarray(wrist).resize((256, 256)))
    else:
        wrist = np.zeros((256, 256, 3), dtype=np.uint8)

    # State: eef_pos(3) + eef_quat(4) = 7-dim
    eef_pos = obs.get("robot0_eef_pos", np.zeros(3))
    eef_quat = obs.get("robot0_eef_quat", np.zeros(4))
    state = np.concatenate([eef_pos, eef_quat]).astype(np.float32)

    # Raw observation dict with LeRobot keys
    raw_obs = {
        "observation.images.image": agentview,
        "observation.images.image2": wrist,
        "observation.state": state,
    }

    # Convert to tensors + normalize images + add batch dim
    obs_frame = prepare_observation_for_inference(raw_obs, device, task=task_language)

    # Preprocess: tokenize language, normalize state
    obs_preprocessed = preprocess(obs_frame)

    # Model inference
    with torch.no_grad():
        action = policy.select_action(obs_preprocessed)

    # Postprocess: unnormalize action
    action = postprocess(action)

    if isinstance(action, torch.Tensor):
        action = action.cpu().numpy().flatten()

    return np.clip(action[:7], -1, 1)


def run_vla_episode(env, vla_policy, preprocess, postprocess, scenario_name,
                    task_language, max_steps, device="cuda", record_video=True):
    """Run one evaluation episode with VLA policy.
    Returns: success (bool), total_reward, steps, frames.
    """
    obs = env.reset()
    if vla_policy is not None:
        vla_policy.reset()  # clear internal action queue

    frames = []
    total_reward = 0.0
    success = False

    for step in range(max_steps):
        action = get_vla_action(vla_policy, preprocess, postprocess,
                                obs, task_language, device=device)

        if record_video and "agentview_image" in obs:
            frames.append(np.flip(obs["agentview_image"], axis=0).copy())

        obs, reward, done, info = env.step(action)
        total_reward += reward

        if env._check_success():
            success = True

        if done:
            break

    if record_video and "agentview_image" in obs:
        frames.append(np.flip(obs["agentview_image"], axis=0).copy())

    return success, total_reward, step + 1, frames


def save_video_file(frames, path, fps=20):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    writer = imageio.get_writer(path, fps=fps)
    for frame in frames:
        writer.append_data(frame)
    writer.close()


def show_video_inline(path):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video controls width="512">'
        f'<source src="data:video/mp4;base64,{data}" type="video/mp4">'
        f'</video>'
    ))


print("VLA evaluation helpers loaded.")

---
## 5. Run Baseline Evaluation on All 6 Scenarios

Same evaluation protocol as the fine-tuned model: 10 episodes per scenario,
with video recording.

In [ ]:
# ============================================================
# RUN BASELINE VLA EVALUATION ON ALL 6 SCENARIOS
# ============================================================

EVAL_EPISODES = 10   # episodes per scenario (same as fine-tuned eval)
EVAL_VIDEO_DIR = "baseline_eval_videos"

eval_results = {}

for scenario_name, cfg in SCENARIOS.items():
    task_lang = TASK_DESCRIPTIONS[scenario_name]
    max_steps = cfg["max_steps"]

    print(f"\n{'='*70}")
    print(f"  EVALUATING BASELINE VLA: {scenario_name}")
    print(f"  Task: {task_lang}")
    print(f"  Episodes: {EVAL_EPISODES}, Max steps: {max_steps}")
    print(f"{'='*70}")

    env = suite.make(**cfg["env_kwargs"])
    ep_results = []

    for ep in range(EVAL_EPISODES):
        np.random.seed(1000 + ep)  # same seeds as fine-tuned eval

        success, reward, steps, frames = run_vla_episode(
            env, vla_policy, vla_preprocess, vla_postprocess,
            scenario_name, task_lang,
            max_steps=max_steps, device=DEVICE, record_video=True,
        )

        status = "SUCCESS" if success else "FAIL"

        ep_results.append({
            "episode": ep, "success": success,
            "reward": reward, "steps": steps,
        })

        print(f"  Episode {ep+1:2d}: {status:7s}  "
              f"reward={reward:.2f}  steps={steps}")

        # Save video with success/fail in filename
        tag = "success" if success else "fail"
        video_path = os.path.join(
            EVAL_VIDEO_DIR, scenario_name,
            f"ep{ep+1:02d}_{tag}.mp4"
        )
        if frames:
            save_video_file(frames, video_path)

    env.close()

    n_success = sum(1 for r in ep_results if r["success"])
    rate = n_success / EVAL_EPISODES

    eval_results[scenario_name] = {
        "n_success": n_success,
        "n_episodes": EVAL_EPISODES,
        "success_rate": rate,
        "episodes": ep_results,
    }

    print(f"\n  {scenario_name} RESULT: {n_success}/{EVAL_EPISODES} ({rate:.0%})")

# ============================================================
# EVALUATION SUMMARY TABLE
# ============================================================
print(f"\n\n{'='*70}")
print(f"  BASELINE VLA EVALUATION SUMMARY (No Fine-Tuning)")
print(f"  Model: {BASELINE_CHECKPOINT}")
print(f"{'='*70}")
print(f"  {'Scenario':<25s} {'Success':>10s} {'Rate':>8s} {'Status':>10s}")
print(f"  {'-'*55}")

total_s, total_e = 0, 0
for name, r in eval_results.items():
    status = "PASS" if r["n_success"] > 0 else "FAIL"
    print(f"  {name:<25s} {r['n_success']:>3d}/{r['n_episodes']:<3d}  "
          f"{r['success_rate']:>7.0%} {status:>10s}")
    total_s += r["n_success"]
    total_e += r["n_episodes"]

print(f"  {'-'*55}")
print(f"  {'OVERALL':<25s} {total_s:>3d}/{total_e:<3d}  {total_s/total_e:>7.0%}")

---
## 6. Display Evaluation Videos

In [ ]:
# ============================================================
# DISPLAY EVALUATION VIDEOS INLINE
# ============================================================
from pathlib import Path

for scenario_name in SCENARIOS.keys():
    video_dir = Path(EVAL_VIDEO_DIR) / scenario_name
    if not video_dir.exists():
        continue

    videos = sorted(video_dir.glob("*.mp4"))
    if not videos:
        continue

    r = eval_results[scenario_name]
    display(HTML(
        f'<h3>{scenario_name} -- '
        f'{r["n_success"]}/{r["n_episodes"]} '
        f'({r["success_rate"]:.0%})</h3>'
    ))

    for video_path in videos:
        name = video_path.stem
        is_success = "success" in name
        color = "green" if is_success else "red"
        status = "SUCCESS" if is_success else "FAIL"

        display(HTML(
            f'<h4 style="color: {color}">{name} -- {status}</h4>'
        ))
        show_video_inline(str(video_path))

---
## 7. Success/Fail Grid Visualization

In [ ]:
# ============================================================
# SUCCESS/FAIL GRID VISUALIZATION
# ============================================================

fig, axes = plt.subplots(1, len(eval_results), figsize=(3 * len(eval_results), 4))
if len(eval_results) == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, eval_results.items()):
    episodes = r["episodes"]
    cols = min(5, len(episodes))
    rows = (len(episodes) + cols - 1) // cols

    for i, ep in enumerate(episodes):
        row = i // cols
        col = i % cols
        color = '#4CAF50' if ep['success'] else '#f44336'
        rect = mpatches.FancyBboxPatch(
            (col, rows - 1 - row), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor='white', linewidth=2
        )
        ax.add_patch(rect)
        ax.text(col + 0.45, rows - 1 - row + 0.45, str(i + 1),
                ha='center', va='center', fontsize=8, color='white', fontweight='bold')

    ax.set_xlim(-0.1, max(cols, 1) + 0.1)
    ax.set_ylim(-0.1, max(rows, 1) + 0.1)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f"{name}\n{r['n_success']}/{r['n_episodes']}", fontsize=10)

plt.suptitle(
    f"Baseline SmolVLA (No Fine-Tuning): {total_s}/{total_e} ({total_s/total_e:.0%})",
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig("baseline_eval_grid.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Save Results & Compare

In [ ]:
# ============================================================
# SAVE BASELINE RESULTS
# ============================================================
os.makedirs("eval_results", exist_ok=True)

baseline_results = {
    "model": "SmolVLA Base (no fine-tuning)",
    "checkpoint": BASELINE_CHECKPOINT,
    "eval_episodes": EVAL_EPISODES,
    "overall_success": total_s,
    "overall_episodes": total_e,
    "overall_rate": total_s / total_e,
    "results": {k: {kk: vv for kk, vv in v.items() if kk != 'episodes'}
                for k, v in eval_results.items()},
    "episodes": {k: v['episodes'] for k, v in eval_results.items()},
}

with open("eval_results/baseline_eval.json", "w") as f:
    json.dump(baseline_results, f, indent=2, default=str)

print(f"Baseline results saved to eval_results/baseline_eval.json")

In [ ]:
# ============================================================
# COMPARISON: BASELINE vs FINE-TUNED (if results available)
# ============================================================
finetuned_path = "eval_results/vla_eval.json"

if os.path.exists(finetuned_path):
    with open(finetuned_path) as f:
        finetuned_results = json.load(f)

    print(f"\n{'='*70}")
    print(f"  COMPARISON: Baseline vs Fine-Tuned SmolVLA")
    print(f"{'='*70}")
    print(f"  {'Scenario':<25s} {'Baseline':>10s} {'Fine-Tuned':>12s} {'Delta':>8s}")
    print(f"  {'-'*58}")

    b_total_s, b_total_e = 0, 0
    f_total_s, f_total_e = 0, 0

    scenarios_both = [s for s in SCENARIOS if s in finetuned_results]

    for name in scenarios_both:
        b_r = eval_results[name]
        f_r = finetuned_results[name]

        b_rate = b_r["success_rate"]
        f_rate = f_r["success_rate"]
        delta = f_rate - b_rate

        sign = "+" if delta > 0 else ""
        print(f"  {name:<25s} {b_rate:>9.0%} {f_rate:>11.0%} {sign}{delta:>7.0%}")

        b_total_s += b_r["n_success"]
        b_total_e += b_r["n_episodes"]
        f_total_s += f_r["n_success"]
        f_total_e += f_r["n_episodes"]

    b_overall = b_total_s / b_total_e if b_total_e > 0 else 0
    f_overall = f_total_s / f_total_e if f_total_e > 0 else 0
    delta_overall = f_overall - b_overall
    sign = "+" if delta_overall > 0 else ""

    print(f"  {'-'*58}")
    print(f"  {'OVERALL':<25s} {b_overall:>9.0%} {f_overall:>11.0%} {sign}{delta_overall:>7.0%}")

    # Bar chart comparison
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(scenarios_both))
    width = 0.35

    b_rates = [eval_results[s]["success_rate"] for s in scenarios_both]
    f_rates = [finetuned_results[s]["success_rate"] for s in scenarios_both]

    bars1 = ax.bar(x - width/2, b_rates, width, label='Baseline (pretrained)', color='#95a5a6')
    bars2 = ax.bar(x + width/2, f_rates, width, label='Fine-tuned on Lift', color='#3498db')

    ax.set_ylabel('Success Rate')
    ax.set_title('SmolVLA: Baseline vs Fine-Tuned')
    ax.set_xticks(x)
    ax.set_xticklabels(scenarios_both, rotation=30, ha='right')
    ax.legend()
    ax.set_ylim(0, 1.05)

    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                        f'{h:.0%}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig("baseline_vs_finetuned.png", dpi=150, bbox_inches='tight')
    plt.show()

else:
    print(f"No fine-tuned results found at {finetuned_path}")
    print("Run the data_collection.ipynb notebook first to generate fine-tuned results.")
    print("\nBaseline-only results:")
    for name, r in eval_results.items():
        print(f"  {name}: {r['n_success']}/{r['n_episodes']} ({r['success_rate']:.0%})")

---
## 9. Save Everything to Google Drive

In [ ]:
# ============================================================
# SAVE BASELINE RESULTS TO GOOGLE DRIVE
# ============================================================
from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/robosuite_vla/baseline"
os.makedirs(DRIVE_DIR, exist_ok=True)

# Save eval results JSON
if os.path.exists("eval_results/baseline_eval.json"):
    shutil.copy2("eval_results/baseline_eval.json",
                 os.path.join(DRIVE_DIR, "baseline_eval.json"))
    print("Saved baseline_eval.json")

# Save evaluation videos
if os.path.exists(EVAL_VIDEO_DIR):
    shutil.copytree(EVAL_VIDEO_DIR,
                    os.path.join(DRIVE_DIR, "videos"),
                    dirs_exist_ok=True)
    print("Saved evaluation videos")

# Save comparison plots
for img in ["baseline_eval_grid.png", "baseline_vs_finetuned.png"]:
    if os.path.exists(img):
        shutil.copy2(img, os.path.join(DRIVE_DIR, img))
        print(f"Saved {img}")

print(f"\nAll baseline results saved to: {DRIVE_DIR}")

---
## Done!

**Expected baseline result:** The pretrained SmolVLA base model should score near 0%
on all robosuite tasks, since it was trained on diverse community data (mostly real-world
robot arms) and has never seen MuJoCo/robosuite environments.

Any improvement in the fine-tuned model over this baseline demonstrates successful
transfer learning from the Lift training data.